## 36122 Python Programming
### (Autumn 2026)
Group Project
----------------------------------

**Student Name and ID:**

Leon Hsu - 26324145

Ishani Bondade - 26147280

Chaemin Jin - 26492722

Niki Miyake - 14742605

**Step 1: Import required libraries**

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import webbrowser
from dotenv import load_dotenv

load_dotenv()

NEWS_KEY = os.getenv("NEWS_API_KEY")
NYT_KEY = os.getenv("NYT_API_KEY")

**Step 2: Web Scraping**

In [ ]:
class NewsScraper:
    def __init__(self, news_key, nyt_key):
        self.news_key = news_key
        self.nyt_key = nyt_key
        self.headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

    def fetch_all_news(self, category):
        
        #Combine results from both APIs
        all_results = []
        
        #Categories that are in NewsAPI
        news_api_categories = ['Business', 'Entertainment', 'General', 'Health', 'Science', 'Sports', 'Technology']
        
        if category in news_api_categories:
            #Fetch from NewsAPI
            news_api_data = self.fetch_news_api(category)
            if isinstance(news_api_data, list):
                all_results.extend(news_api_data)

        #Fetch from NYT API
        nyt_api_data = self.fetch_nyt_api(category)
        if isinstance(nyt_api_data, list):
            all_results.extend(nyt_api_data)
            
        return all_results
    
    def fetch_news_api(self, category):
        url = "https://newsapi.org/v2/top-headlines"

        #Request for API
        params = {
            'apikey': self.news_key,
            'category': category.lower(),
            'language': 'en',
            'pageSize': 10
        }

        try:
            response = requests.get(url, params=params)
            articles = response.json().get('articles', [])
            return [{'title': a['title'], 'source': a['source']['name'], 'url': a['url'], 'summary': self.scrape_article(a['url'])} for a in articles]
        
        except Exception as e:
            return f"News API Error!: {str(e)}"
        
    def fetch_nyt_api(self, category):
        mapping = {
            "General": "world",
            "Entertainment": "arts",
            "Business": "business",
            "Technology": "technology",
            "Science": "science",
            "Health": "health",
            "Sports": "sports",
            "Fashion": "fashion",
            "Food": "dining",
            "Travel": "travel",
            "RealEstate": "realestate",
            "Politics": "politics"
        }
        nyt_cat = mapping.get(category, "world")
        url = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

        #Request for API
        params = {
            'q': nyt_cat,
            'api-key': self.nyt_key
        }

        try:
            response = requests.get(url, params=params)
            data = response.json()
            docs = data.get('response', {}).get('docs', [])
            
            if not docs:
                return []
            
            return [{'title': d['headline']['main'], 'source': 'New York Times', 'url': d['web_url'], 'summary': d.get('abstract', 'No summary available')} for d in docs[:10]]
        
        except Exception as e:
            print(f"NYT Error!: {str(e)}")
            return []

    def scrape_article(self, url):
        try:
            response = requests.get(url, headers=self.headers, timeout=5)
            soup = BeautifulSoup(response.content, 'html.parser')
            # Extract the first paragraph from the article
            paragraphs = soup.find_all('p')
            valid_paragraphs = []
            garbage_words = ['advertisement', 'subscribe', 'sign up', 'cookie', 'privacy policy', 'terms of service', 'javascript', 'browser version', 'enable cookies', 'support for css']
            
            for p in paragraphs:
                text = p.get_text().strip()
                if len(text) < 50:
                    continue
                if any(word in text.lower() for word in garbage_words):
                    continue
                valid_paragraphs.append(text)
                
            if valid_paragraphs:
                return "\n\n".join(valid_paragraphs[:5])
            return "No summary available!"
            
        except:
            return "Summary extraction failed!"

**Test the Scrapping**

In [ ]:
scraper = NewsScraper(NEWS_KEY, NYT_KEY)
topic = "Business"
results = scraper.fetch_all_news(topic)
print(f"-----News for '{topic}'-----")
if isinstance(results, list) and len(results) > 0:
    for i, item in enumerate(results[:5], 1): # Showing top 5 to see both sources
        print(f"{i}. [{item['source']}]")
        print(f"   Title: {item['title']}")
        clean_summary = str(item['summary']).split('\n')[0]
        print(f"   Summary: {clean_summary[:100]}...")
        print(f"   Link: {item['url']}")
        print("-" * 30)
else:
    print("No results found. Check if your API keys are correct or if there is a naming error in the class.")

**Step 3: GUI**

In [ ]:
class NewsApp:
    def __init__(self, root, news_engine):
        self.root = root
        self.news_engine = news_engine
        self.root.title("News Scrapper")
        self.root.geometry("800x600")

        #Create the Menu
        tk.Label(root, text="Select the news category: ", font=("Times New Roman", 14, "bold")).pack(pady=10)
        self.category_list = ['Business', 'Entertainment', 'General', 'Health', 'Science', 'Sports', 'Technology', 'Fashion', 'Food', 'Travel', 'RealEstate', 'Politics']
        self.dropdown = ttk.Combobox(root, values=self.category_list, state="readonly")
        self.dropdown.set("General")
        self.dropdown.pack()

        #Create the button to fetch news
        tk.Button(root, text="Search News", command=self.handle_search, bg="blue", fg="white", font=("Times New Roman", 14, "bold")).pack(pady=10)

        #Display
        self.display_frame = tk.Frame(root)
        self.display_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=10)
        
        self.scrollbar = tk.Scrollbar(self.display_frame)
        self.scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        self.display = tk.Text(self.display_frame, wrap=tk.WORD, padx=15, pady=15, font=("Times New Roman", 12), yscrollcommand=self.scrollbar.set)
        self.display.pack(fill=tk.BOTH, expand=True)
        self.scrollbar.config(command=self.display.yview)
        
        self.display.tag_config("bold_blue", foreground="blue", underline=True, font=("Times New Roman", 12, "bold"))
        self.display.tag_config("source_tag", foreground="green", font=("Times New Roman", 10, "italic"))
        self.display.tag_config("hint", foreground="purple", font=("Times New Roman", 10, "italic"))
        
        self.display.config(state="disabled")
        
    def handle_search(self):
        selected_category = self.dropdown.get()
        self.display.config(state="normal")
        self.display.delete("1.0", tk.END)
        self.display.insert(tk.END, "Loading...\n")
        self.root.update()
        
        
        results = self.news_engine.fetch_all_news(selected_category)
        self.current_article = results

        self.display.delete("1.0", tk.END)
        
        if isinstance(results, str):
            messagebox.showerror("Error", results)
        elif not results:
            self.display.insert(tk.END, "No articles found for this category. Try another category!")

        #if not self.current_article or isinstance(self.current_article, str):
            #messagebox.showerror("Error", self.current_article)

        else:
            for i, news in enumerate(self.current_article):
                tag_name = f"article_{i}"
                self.display.insert(tk.END, f"Title: {news['title']}\n", tag_name)
                self.display.tag_config(tag_name, foreground="blue", underline=True, font=("Times New Roman", 12, "bold"))
                
                self.display.insert(tk.END, f"Source: {news['source'].upper()}\n", "source_tag")
                #self.display.tag_config("source_tag", foreground="green", font=("Times New Roman", 10, "italic"))

                raw_summary = news.get('summary', 'No summary available!')
                preview_para = str(raw_summary).split('\n\n')[0]
                self.display.insert(tk.END, f"Overview: {preview_para}\n")
                
                self.display.insert(tk.END, "Double-click the title to read more...\n\n", "hint")
                self.display.insert(tk.END, "="*50 + "\n\n")
                
                #Clickable title
                self.display.tag_bind(tag_name, "<Double-1>", lambda e, idx=i: self.show_full_article(idx))
                self.display.tag_bind(tag_name, "<Enter>", lambda e: self.display.config(cursor="hand2"))
                self.display.tag_bind(tag_name, "<Leave>", lambda e: self.display.config(cursor=""))

        self.display.config(state="disabled")

    def show_full_article(self, index):
        #New pop up window to display the complete article
        article = self.current_article[index]
        new_window = tk.Toplevel(self.root)
        new_window.title(article['title'])
        new_window.geometry("800x500")

        #Header Information
        tk.Label(new_window, text=article['title'], font=("Times New Roman", 16, "bold"), wraplength=600, justify="left", fg="#1227AF").pack(pady=10, padx=20)

        #Open article in browser if the article is not scrapped
        tk.Button(new_window, text="Open in Browser", bg="orange", fg="black", font=("Times New Roman", 16, "bold"), command=lambda: webbrowser.open(article['url'])).pack(pady=5)

        #Make it scrollable
        text = tk.Text(new_window, wrap=tk.WORD, padx=20, pady=25, font=("Times New Roman", 12))
        text.pack(fill=tk.BOTH, expand=True)
        
        raw_content = article.get('summary', '')
        
        if not raw_content or str(raw_content).strip().lower() in ["none", "","No summary available!"]:
            display_body = (
                "Notice: Content extraction restricted.\n\n"
                "The full content of this article is currently unavailable for direct preview.\n\n"
                "Please click the 'Open in Browser' button to read the complete article."
            )
        else:
            display_body = raw_content
            
        full_content = f"Source: {article['source']}\n"
        full_content += f"URL: {article['url']}\n"
        full_content += "-"*50 + "\n\n"
        full_content += display_body
        
        text.insert("1.0", full_content)
        text.config(state="disabled") #This makes the text read-only

**Step 4: Launch the NEWS APP**

In [ ]:
if __name__ == "__main__":
    try:
        main = tk.Tk()
        news_engine = NewsScraper(NEWS_KEY, NYT_KEY)
        app = NewsApp(main, news_engine)
        main.mainloop()

    except Exception as e:
        print(f"App closed. Error: {e}")